# Gateway — Agent-Analyse

Tier-1 eval (#388). Tests proaktive Push via `send_notification` Tool +
direkter CommunicationGateway-Smoke-Test mit InMemory-Stores.
Keine echten Telegram/Teams-Credentials nötig.

## 1. Setup

In [ ]:
import os, sys, importlib
from pathlib import Path

PYTF_ROOT = Path.cwd()
if PYTF_ROOT.name == 'notebooks':
    PYTF_ROOT = PYTF_ROOT.parent
os.chdir(PYTF_ROOT)
for p in [str(PYTF_ROOT / 'src'), str(PYTF_ROOT / 'notebooks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from dotenv import load_dotenv
    load_dotenv(PYTF_ROOT / '.env')
except ImportError:
    pass

try:
    import nest_asyncio; nest_asyncio.apply()
except ImportError:
    pass

import logging, structlog
logging.basicConfig(level=logging.WARNING, format='%(levelname)s %(name)s | %(message)s')
structlog.configure(wrapper_class=structlog.make_filtering_bound_logger(logging.WARNING))

import analysis_lib
importlib.reload(analysis_lib)
import analysis_lib as alib
print('analysis_lib OK -', len(alib.list_evaluators()), 'evaluators')

In [ ]:
from taskforce.application.factory import AgentFactory
from taskforce.application.executor import AgentExecutor

factory = AgentFactory()
executor = AgentExecutor(factory=factory)
alib.disable_post_mission_learning(executor)
print('factory + executor ready')

## 2. In-Memory Gateway aufbauen

Mock-Setup: `CountingSender` zählt outbound calls, `InMemoryGatewayConversationStore`
+ `InMemoryRecipientRegistry` halten den State. Keine echten Telegram/Teams-Credentials
nötig. Der Agent wird mit `send_notification` Tool ausgestattet.

In [ ]:
from typing import Any
from taskforce.application.gateway import CommunicationGateway
from taskforce.core.domain.gateway import InboundMessage, NotificationRequest
from taskforce.infrastructure.communication.gateway_conversation_store import (
    InMemoryGatewayConversationStore,
)
from taskforce.infrastructure.communication.recipient_registry import (
    InMemoryRecipientRegistry,
)

WORKDIR = Path('.taskforce_gateway_analysis')
WORKDIR.mkdir(exist_ok=True)

class CountingSender:
    def __init__(self, channel: str = 'telegram'):
        self._channel = channel
        self.sent: list[tuple[str, str, dict | None]] = []
    @property
    def channel(self) -> str:
        return self._channel
    async def send(self, *, recipient_id: str, message: str,
                   metadata: dict[str, Any] | None = None):
        self.sent.append((recipient_id, message, metadata))
    async def send_file(self, **kw):
        pass

tg_sender = CountingSender('telegram')
conv_store = InMemoryGatewayConversationStore()
registry = InMemoryRecipientRegistry()

# Wire a CommunicationGateway on the factory so send_notification tool
# has a real backend (otherwise it errors 'gateway not configured').
gateway = CommunicationGateway(
    executor=executor,
    conversation_store=conv_store,
    recipient_registry=registry,
    outbound_senders={'telegram': tg_sender},
)
factory.set_gateway(gateway)

# Register the notebook's default recipient so send_notification can
# resolve it (patch_notification_defaults sets it on the tool, but the
# registry must know about it too).
await registry.register(channel='telegram', user_id='nb-user',
                        reference={'conversation_id': 'nb-conv'})

GATEWAY_TOOLS = ['send_notification', 'ask_user', 'python']
GATEWAY_PROMPT = (
    'Du bist ein Assistent mit Push-Channel. send_notification(channel, '
    'recipient_id, message) für proaktive Nachrichten. Genau eine Nachricht pro '
    'Auftrag, nicht mehrere.'
)

async def build_gateway_agent():
    a = await factory.create_agent(
        system_prompt=GATEWAY_PROMPT, tools=GATEWAY_TOOLS,
        persistence={'type': 'file', 'work_dir': str(WORKDIR)},
        work_dir=str(WORKDIR),
        planning_strategy='native_react', max_steps=4,
    )
    alib.patch_anti_compression(a)
    # Default recipient so send_notification doesn't refuse
    alib.patch_notification_defaults(a, default_channel='telegram',
                                     default_recipient_id='nb-user')
    return a, len(GATEWAY_PROMPT)

BUILD_AGENT = build_gateway_agent
_smoke, _chars = await BUILD_AGENT()
AVAILABLE_TOOLS = set(_smoke.tools.keys())
print(f'tools={sorted(AVAILABLE_TOOLS)}, sender ready')
await _smoke.close()


## 3. Szenarien laden

Aus `scenarios/gateway.yaml`.

In [ ]:
all_scenarios = alib.load_scenarios('notebooks/scenarios/gateway.yaml')
eligible = alib.filter_scenarios(all_scenarios, AVAILABLE_TOOLS)
print(f'Total: {len(all_scenarios)}, Eligible: {len(eligible)}')
for s in eligible:
    print(f'  - {s.id:35s} ({s.category}/{s.difficulty})')

## 4. Einzelnes Szenario (Detail-Lauf)

In [ ]:
TARGET_ID = 'gateway-push-simple'  # change me
target = next((s for s in eligible if s.id == TARGET_ID), None) or eligible[0]
print(f'Target: {target.id}')
print(f'Mission: {target.mission}')
print(f'Hidden intent: {target.hidden_intent}')

In [ ]:
agent, sys_chars = await BUILD_AGENT()
rec = await alib.run(
    executor, agent, target.mission,
    project_root=WORKDIR, snapshot_subdirs=(),
    initial_system_prompt_chars=sys_chars, silent=False,
)
# Stash MCP tool names if any (no-op for non-MCP notebooks)
rec.extra['mcp_tool_names'] = list(_mcp_tool_names_of(agent)) if '_mcp_tool_names_of' in dir() else []
alib.print_summary(rec)

## 5. Reports

In [ ]:
alib.print_tool_stats(rec)
print()
alib.print_tool_results(rec, head=15)

In [ ]:
card = alib.score_rule_based(rec, target)
print(f'=== Rule-based ({"PASS" if card.passed else "FAIL"}) ===')
alib.print_feature_checks(card)
if card.details:
    print('Details:')
    for d in card.details:
        print(f'  - {d}')

In [ ]:
await agent.close()
_ = alib.plot_tool_frequencies(rec, title=f'Tool-Aufrufe: {target.id}')

## 6. Batch-Lauf

In [ ]:
results = await alib.run_scenarios(
    executor, BUILD_AGENT, eligible,
    project_root=WORKDIR, snapshot_subdirs=(),
    reset_dirs_before_each=(),
    repeats=1, progress=True,
)  
print()
alib.print_scenario_summary(results)

In [ ]:
_ = alib.plot_scenario_matrix(results, metric='passed', title='Pass/Fail')
_ = alib.plot_scenario_matrix(results, metric='tool_calls', title='Tool calls')

## 7. CommunicationGateway direkt (ohne Agent)

Testet inbound-handling + notification-API ohne LLM-Calls — schnell und deterministisch.

In [ ]:
from taskforce.core.domain.models import ExecutionResult

class FakeExecutor:
    def __init__(self):
        self.calls = []
    async def execute_mission(self, **kwargs: Any) -> ExecutionResult:
        self.calls.append(kwargs)
        return ExecutionResult(
            session_id=kwargs.get('session_id', 'fake-session'),
            status='completed',
            final_message=f"echo: {kwargs.get('mission', '?')[:50]}",
        )

fake_exec = FakeExecutor()
gw = CommunicationGateway(
    executor=fake_exec,
    conversation_store=InMemoryGatewayConversationStore(),
    recipient_registry=InMemoryRecipientRegistry(),
    outbound_senders={'telegram': CountingSender('telegram')},
)

# Test: inbound message routes to executor and produces an outbound
msg = InboundMessage(channel='telegram', conversation_id='chat-1',
                     message='hi', sender_id='user-1')
resp = await gw.handle_message(msg)
print(f'reply={resp.reply!r}')
print(f'executor calls: {len(fake_exec.calls)}')
outbound_sender = gw._outbound_senders['telegram']
print(f'outbound sent : {len(outbound_sender.sent)} message(s)')

## Ideen für weitere Experimente

Siehe TODOs in der Scenario-YAML.

In [ ]:
print('done')